In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 벡터 DB : Chroma vs. Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB, collection단위로 저장
- Pinecone : 클라우드 vector DB, index단위로 저장
(http://www.pinecone.io)에서 api key 받아 .env추가(PINECONE_API_KEY등록)

# 0. 패키지 설치

In [2]:
%pip install -q pinecone langchain-pinecone # --no-warn-script-location

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


# 1. knowledge base 구성을 위한 데이터 생성

In [3]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docx/소득세법(법률)(제21221호)(20260701).docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   # 문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n", "\n", " ", ""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n", "\n", " ", ""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

<timed exec>:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


CPU times: total: 2.88 s
Wall time: 2.88 s


180

In [1]:
# 임베딩 객체
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding = OpenAIEmbeddings(model = 'text-embedding-3-large')

In [2]:
# # 임베딩 객체
# from dotenv import load_dotenv
# from langchain_upstage import UpstageEmbeddings
# load_dotenv()
# embedding = UpstageEmbeddings(model = 'solar-embedding-1-large-passage')

In [3]:
len(embedding.embed_query("샘플"))

3072

In [7]:
# 모델마다 코사인유사도를 쓰는지 document
# https://console.upstage.ai/docs/capabilities/embed
# https://developers.openai.com/api/docs/guides/embeddings

In [6]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os
pc = Pinecone(
    # api_key=os.getenv('PINECONE_API_KEY')
)
# # 데이터를 처음 저장할 때
# index_name = 'tax-index-openai'
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )
#업로드된 벡터 DB를 load할 때
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name
)

CPU times: total: 250 ms
Wall time: 1.4 s


In [10]:
#%pip install -q ipywidgets

# 2. 답변 생성을 위한 Retrieval

In [7]:
query = "연봉 5천만원인 직장인은 소득세를 얼마낼까요?"
retrieved_docs = database.similarity_search(query, k=3) # 기본k값은 4

In [8]:
retrieved_doc = "\n--\n".join([doc.page_content.replace('\n\n','\n') for doc in retrieved_docs])
#print(retrieved_doc)

In [12]:
retrieved_doc

'제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31., 2024. 12. 31., 2025. 10. 1., 2025. 12. 23.>\n1. 「공익신탁법」에 따른 공익신탁의 이익\n2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득\n가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득\n나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다). 이 경우 주택 수의 계산 및 주택임대소득의 산정 등 필요한 사항은 대통령령으로 정한다.\n다. 대통령령으로 정하는 농어가부업소득\n라. 대통령령으로 정하는 전통주의 제조에서 발생하는 소득\n마. 조림기간 5년 이상인 임지(林地)의 임목(林木)의 벌채 또는 양도로 발생하는 소득으로서 연 3천만원 이하의 금액. 이 경우 조림기간 및 세액의 계산 등 필요한 사항은 대통령령으로 정한다.\n바. 대통령령으로 정하는 작물재배업에서 발생하는 소득\n사. 대통령령으로 정하는 어로어업 또는 양식어업에서 발생하는 소득\n3. 근로소득과 퇴직소득 중 다음 각 목의 어느 하나에 해당하는 소득\n가. 대통령령

In [13]:
'55조' in retrieved_doc

False

# 3. 답변생성

In [9]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-nano')

In [10]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(model = 'solar-pro2')

In [11]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다. [context]를 참고해서 사용자 질문에 답하세요. [context]는 다음과 같아요
{retrieved_doc}
- 질문 : {query}"""

In [16]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

연봉 5천만원인 직장인의 소득세 계산은 다음과 같은 단계를 거쳐 진행됩니다. 다만, 정확한 세액은 개별 공제 항목(의료비, 교육비, 기부금 등)에 따라 달라질 수 있으므로, 여기서는 **기본적인 계산 방식**을 안내드립니다.

---

### **1. 근로소득 금액 계산**
- **연봉 5,000만원**에서 **근로소득 공제**를 적용합니다.  
  - 근로소득공제율: **5,000만원 × 70% + 1,500만원 × 40%** (단, 2024년 기준 최대 1,500만원 한도)  
  - **공제액**: 5,000만원 × 70% = **3,500만원**  
  - **근로소득 금액**: 5,000만원 - 3,500만원 = **1,500만원**

> ※ 근로소득공제 한도: 2024년 기준 연봉 7,000만원 이하 근로자는 70% 공제.

---

### **2. 종합소득과세표준 계산**
근로소득 금액에서 **인적 공제** 등 기본 공제를 적용합니다.  
- **기본공제**: 본인 150만원 + 배우자(있는 경우) 150만원 + 부양가족(1인당 150만원) 등  
  - 예시: 본인만 있는 경우 **150만원** 공제  
- **과세표준**: 1,500만원(근로소득) - 150만원(기본공제) = **1,350만원**

---

### **3. 산출세액 계산 (종합소득세율 적용)**
- **누진세율** 적용 (2024년 기준):  
  ```
  1,200만원 이하: 6%  
  1,200만원 초과 ~ 4,600만원 이하: 15%  
  ```
  - 계산:  
    - 1,200만원 × 6% = 72만원  
    - (1,350만원 - 1,200만원) × 15% = 22.5만원  
    - **산출세액**: 72만원 + 22.5만원 = **94.5만원**

---

### **4. 세액공제 및 감면 적용**
- **근로소득 세액공제**: 산출세액의 55% (최대 72만원 한도)  
  - 94.5만원 × 55% = 51.975만원 → **51.98만원** 공